In [26]:
import pandas as pd
import random

# Define the CSV file path and name
csv_file_path = '/content/napi_tucat_novenyi_adatbazis_napi_minimum_oszlopokkal.csv'

# Define the required columns using their Hungarian names as they appear in the CSV
required_columns_hungarian = [
    'Élelmiszer_magyar',
    'Kategória',
    'DailyDozen',
    'Serving_g',
    'Daily_serving_target',
    'Serving_description_hu',
    'Calories_per_serving',
    'Protein_per_serving',
    'Fiber_per_serving',
    'Zsír_g_100g',        # Corrected Fat column name
    'Szénhidrát_g_100g' # Corrected Carbohydrate column name
]

# Define a mapping from Hungarian column names to standardized English names for internal use
column_rename_map = {
    'Élelmiszer_magyar': 'Hungarian',
    'Kategória': 'Category',
    'Zsír_g_100g': 'Fat_g_per_serving',          # Mapping for Fat
    'Szénhidrát_g_100g': 'Carbohydrate_g_per_serving' # Mapping for Carbohydrate
}

# --- 1. Read the CSV file using pandas ---
try:
    df = pd.read_csv(csv_file_path)
    print(f"CSV file '{csv_file_path}' loaded successfully.")
except FileNotFoundError:
    print(f"Error: The file '{csv_file_path}' was not found. Please ensure it is uploaded to Colab and the filename is correct.")
    df = pd.DataFrame() # Create an empty DataFrame to prevent subsequent errors
except Exception as e:
    print(f"An error occurred while reading the CSV file: {e}")
    df = pd.DataFrame() # Create an empty DataFrame in case of other reading errors

# --- 2. Check if all required columns are present (using Hungarian names) ---
if not df.empty:
    missing_columns = [col for col in required_columns_hungarian if col not in df.columns]
    if missing_columns:
        print(f"Error: The following required columns are missing from the CSV: {', '.join(missing_columns)}.")
        print("Please ensure the CSV file has all the necessary columns with their exact names.")
        print("Available columns in the CSV are:")
        print(df.columns.tolist()) # Print all columns to identify correct names
        df = pd.DataFrame() # Clear DataFrame if columns are missing
    else:
        print("All required columns are present.")
        # --- Rename columns for internal consistency ---
        df.rename(columns={k: v for k, v in column_rename_map.items() if k in df.columns}, inplace=True)
        print("Relevant columns renamed for processing.")


CSV file '/content/napi_tucat_novenyi_adatbazis_napi_minimum_oszlopokkal.csv' loaded successfully.
All required columns are present.
Relevant columns renamed for processing.


### Oszloponkénti adatmennyiség ellenőrzése

Ez a szakasz egy áttekintést nyújt az adatkeret (`df`) oszlopainak telítettségéről, megjelenítve a kitöltött és hiányzó értékek számát, valamint azok százalékos arányát.

In [27]:
if not df.empty:
    # Count non-null values per column
    non_null_counts = df.count()

    # Get total number of rows
    total_rows = df.shape[0]

    # Calculate null counts and percentage of non-null values
    null_counts = total_rows - non_null_counts
    non_null_percentage = (non_null_counts / total_rows) * 100

    # Create a DataFrame for better display
    column_data_summary = pd.DataFrame({
        'Kitöltött adatok száma': non_null_counts,
        'Hiányzó adatok száma': null_counts,
        'Kitöltöttség (%)': non_null_percentage.round(2)
    })

    # Sort the summary in descending order by 'Kitöltött adatok száma'
    column_data_summary = column_data_summary.sort_values(by='Kitöltött adatok száma', ascending=False)

    print("Oszloponkénti adatmennyiség és kitöltöttségi összefoglaló (csökkenő sorrendben):")
    display(column_data_summary)
else:
    print("Az adatkeret üres, nem lehet oszlopadatokat ellenőrizni.")


Oszloponkénti adatmennyiség és kitöltöttségi összefoglaló (csökkenő sorrendben):


,Kitöltött adatok száma,Hiányzó adatok száma,Kitöltöttség (%)
Élelmiszer_angol,108,0,100.00
Hungarian,108,0,100.00
Category,108,0,100.00
DailyDozen,108,0,100.00
Kalória_kcal_100g,108,0,100.00
Fehérje_g_100g,108,0,100.00
Rost_g_100g,108,0,100.00
Carbohydrate_g_per_serving,108,0,100.00
Fat_g_per_serving,108,0,100.00
Szezon,108,0,100.00


In [7]:
# Proceed only if DataFrame is not empty and has all required columns (after potential renaming)
if not df.empty:
    # --- 3. Filter for rows where 'DailyDozen' value is 'Yes' ---
    if 'DailyDozen' in df.columns:
        daily_dozen_df = df[df['DailyDozen'].astype(str).str.lower() == 'yes'].copy()
        if daily_dozen_df.empty:
            print("No items found with 'DailyDozen' value 'Yes'. Please check the data or the 'DailyDozen' column values.")
        else:
            print(f"Found {len(daily_dozen_df)} items marked as 'DailyDozen'.")
    else:
        print("Error: 'DailyDozen' column not found. This should not happen if previous checks passed.")
        daily_dozen_df = pd.DataFrame()

    if not daily_dozen_df.empty:
        # --- 4. Group by 'Category' and determine Daily_serving_target ---
        # --- 5. Randomly select items based on Daily_serving_target ---
        final_suggestion_list = []
        category_counts = {} # To store counts for each category

        if 'Category' in daily_dozen_df.columns and 'Daily_serving_target' in daily_dozen_df.columns:
            for category, group in daily_dozen_df.groupby('Category'):
                # Get the Daily_serving_target for the current category
                # Assuming Daily_serving_target is consistent within a category for DailyDozen items
                try:
                    num_servings = int(group['Daily_serving_target'].iloc[0])
                except (ValueError, IndexError):
                    print(f"Warning: Could not determine 'Daily_serving_target' for category '{category}'. Skipping.")
                    continue

                if num_servings <= 0:
                    print(f"No servings required for category '{category}'. Skipping.")
                    continue

                # Determine if sampling with replacement is needed
                replace_needed = num_servings > len(group)
                if replace_needed:
                    print(f"Warning: Not enough unique items in category '{category}' ({len(group)} items) to meet target ({num_servings} servings). Allowing repetitions.")

                # Randomly sample items for the current category
                try:
                    sampled_items = group.sample(n=num_servings, replace=replace_needed, random_state=None)
                    final_suggestion_list.append(sampled_items)
                    category_counts[category] = num_servings # Store the actual number of servings selected
                except ValueError as e:
                    print(f"Error sampling for category '{category}': {e}. Skipping.")

            if final_suggestion_list:
                daily_suggestion = pd.concat(final_suggestion_list).reset_index(drop=True)
                print(f"Generated daily suggestions across {len(category_counts)} categories.")
            else:
                print("Could not generate any daily suggestions from the categories. Exiting.")
                daily_suggestion = pd.DataFrame()
        else:
            print("Error: 'Category' or 'Daily_serving_target' column not found. Cannot group and sample.")
            daily_suggestion = pd.DataFrame()

        if not daily_suggestion.empty:
            # --- 7. Create a result table with specific columns ---
            result_columns = [
                'Category', # Renamed from 'DailyDozen_kategória'
                'Hungarian', # Renamed from 'Élelmiszer_magyar'
                'Serving_description_hu',
                'Serving_g',
                'Calories_per_serving',
                'Protein_per_serving',
                'Fiber_per_serving'
            ]

            # Ensure these columns exist before selecting them
            available_result_columns = [col for col in result_columns if col in daily_suggestion.columns]
            final_suggestion_table = daily_suggestion[available_result_columns]

            print("\n--- Your Customized Daily Dozen Suggestion ---")
            display(final_suggestion_table)

            # --- 8. Calculate totals for all selected items ---
            total_calories = final_suggestion_table['Calories_per_serving'].sum()
            total_protein = final_suggestion_table['Protein_per_serving'].sum()
            total_fiber = final_suggestion_table['Fiber_per_serving'].sum()

            # --- 9. Display the daily suggestion and totals neatly ---
            print("\n--- Nutritional Summary for Today's Daily Dozen ---")
            print(f"Total Calories: {total_calories:.2f} kcal")
            print(f"Total Protein: {total_protein:.2f} g")
            print(f"Total Fiber: {total_fiber:.2f} g")

            # --- 9.b. Display category-wise counts ---
            print("\n--- Servings Selected Per Category ---")
            for category, count in category_counts.items():
                print(f"{category}: {count} serving(s)")
        else:
            print("Could not generate daily suggestions. Please check the data and filtering steps.")
    else:
        print("No daily dozen items to process. Exiting.")
else:
    print("DataFrame is empty or missing required columns. Cannot proceed with Daily Dozen generation.")


Found 108 items marked as 'DailyDozen'.
Generated daily suggestions across 10 categories.

--- Your Customized Daily Dozen Suggestion ---


,Category,Hungarian,Serving_description_hu,Serving_g,Calories_per_serving,Protein_per_serving,Fiber_per_serving
0,Bogyós gyümölcsök,Málna,1 adag bogyós gyümölcs: kb. 75 g,75,39.00,0.90,4.88
1,Diófélék és magvak,Pekándió,1 adag dióféle vagy mag: kb. 30 g,30,207.30,2.76,2.88
2,Egyéb zöldségek,Zeller,1 adag egyéb zöldség: kb. 100 g,100,42.00,1.50,1.80
3,Egyéb zöldségek,Sütőtök,1 adag egyéb zöldség: kb. 100 g,100,26.00,1.00,0.50
4,Fűszerek,Kurkuma,1 adag fűszer: kb. 2 g,2,6.24,0.19,0.45
5,Gyümölcsök,Citrom,1 adag gyümölcs: kb. 150 g,150,43.50,1.65,4.20
6,Gyümölcsök,Alma,1 adag gyümölcs: kb. 150 g,150,78.00,0.45,3.60
7,Gyümölcsök,Görögdinnye,1 adag gyümölcs: kb. 150 g,150,45.00,0.90,0.60
8,Hüvelyesek,Sárgaborsó,1 adag hüvelyes: kb. 100 g,100,118.00,8.30,8.30
9,Hüvelyesek,Edamame,1 adag hüvelyes: kb. 100 g,100,121.00,11.90,5.20



--- Nutritional Summary for Today's Daily Dozen ---
Total Calories: 1505.14 kcal
Total Protein: 63.74 g
Total Fiber: 57.63 g

--- Servings Selected Per Category ---
Bogyós gyümölcsök: 1 serving(s)
Diófélék és magvak: 1 serving(s)
Egyéb zöldségek: 2 serving(s)
Fűszerek: 1 serving(s)
Gyümölcsök: 3 serving(s)
Hüvelyesek: 3 serving(s)
Keresztesvirágú zöldségek: 1 serving(s)
Lenmag: 1 serving(s)
Teljes értékű gabonák: 3 serving(s)
Zöld levelesek: 2 serving(s)


In [8]:
import pandas as pd
import os

# --- 1. Automatikusan keresse meg az első feltöltött CSV fájlt ---
files_in_content = os.listdir('/content/')
csv_files = [f for f in files_in_content if f.endswith('.csv')]

if not csv_files:
    print("Nincs CSV fájl feltöltve.")
else:
    csv_file_name = csv_files[0]
    csv_file_path = os.path.join('/content/', csv_file_name)

    # --- 2. Olvassa be a CSV fájlt pandas segítségével ---
    try:
        df = pd.read_csv(csv_file_path)

        # --- 3. Számolja ki és jelenítse meg a számadatokat ---
        rows_count = df.shape[0]
        cols_count = df.shape[1]
        total_cells_count = rows_count * cols_count

        print(f"Sorok száma: {rows_count}")
        print(f"Oszlopok száma: {cols_count}")
        print(f"Összes cella száma: {total_cells_count}")

    except Exception as e:
        print(f"Hiba történt a fájl '{csv_file_name}' feldolgozásakor: {e}")

Sorok száma: 108
Oszlopok száma: 33
Összes cella száma: 3564


In [9]:
import requests

# Open Food Facts API végpont (példa egy termék keresésére vagy egy nem létező végpontra)
# Használhatunk egy olyan végpontot, ami valószínűleg nem létezik, vagy általános keresést
api_url = "https://world.openfoodfacts.org/api/v2/product/737628064502.json" # Példa egy termékre
# api_url = "https://world.openfoodfacts.org/api/v2/this_endpoint_does_not_exist" # Vagy egy nem létező végpont

print(f"Próbálkozás az Open Food Facts API elérésével: {api_url}\n")

try:
    response = requests.get(api_url, timeout=10) # 10 másodperces időtúllépés beállítása

    print(f"API válaszkód: {response.status_code}")

    if response.status_code == 503:
        print("Hiba: Az Open Food Facts API átmenetileg nem elérhető (503 Service Unavailable).")
        print(f"Válasz tartalom: {response.text[:200]}...") # Az első 200 karakter a válaszból
    elif not response.ok: # Ellenőrizzük, hogy a válasz sikeres-e (200-299 közötti státuszkód)
        print(f"Hiba történt az API hívás során. Státuszkód: {response.status_code}")
        print(f"Hibaüzenet (ha van): {response.text[:200]}...")
    else:
        print("API hívás sikeresnek tűnik (bár a cél a hibakezelés volt). Státuszkód: 200 OK.")
        print(f"Válasz tartalom (részlet): {response.json().get('product', {}).get('product_name', 'Nincs terméknév')}...")

except requests.exceptions.ConnectionError:
    print("Hiba: Az Open Food Facts API nem elérhető, vagy nincs internetkapcsolat.")
    print("Lehet, hogy a szerver leállt, vagy hálózati probléma van.")
except requests.exceptions.Timeout:
    print("Hiba: Az API hívás időtúllépés miatt meghiúsult. A szerver túl sokáig tartott a válaszadással.")
except Exception as e:
    print(f"Váratlan hiba történt: {e}")


Próbálkozás az Open Food Facts API elérésével: https://world.openfoodfacts.org/api/v2/product/737628064502.json

API válaszkód: 403
Hiba történt az API hívás során. Státuszkód: 403
Hibaüzenet (ha van): <html>
<head><title>403 Forbidden</title></head>
<body>
<center><h1>403 Forbidden</h1></center>
<hr><center>nginx</center>
</body>
</html>
...


In [19]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

diagram_tipus = widgets.Dropdown(
    options=["Category", "Szezon"],
    value="Category",
    description="Megjelenítés:"
)

def rajzol(oszlop):
    clear_output(wait=True)

    # Ensure the DataFrame is not empty before attempting to plot
    if not df.empty and oszlop in df.columns:
        counts = df[oszlop].value_counts()

        counts.plot(kind="bar", figsize=(9, 5))
        plt.title(f"{oszlop} szerinti eloszlás")
        plt.xlabel(oszlop)
        plt.ylabel("Darabszám")
        plt.xticks(rotation=45)
        plt.tight_layout()
        # Removed plt.show() as it can interfere with ipywidgets.Output
    elif df.empty:
        print("Az adatkeret üres, nem lehet diagramot rajzolni.")
    else:
        print(f"A kiválasztott oszlop ('{oszlop}') nem található az adatkeretben.")

# The interactive output will display the plot based on dropdown selection
output = widgets.interactive_output(
    rajzol,
    {"oszlop": diagram_tipus}
)

# Display the dropdown widget itself and the output widget containing the plot
display(diagram_tipus, output)


Dropdown(description='Megjelenítés:', options=('Category', 'Szezon'), value='Category')

Output()

### Átlagos tápanyag-összetétel kategóriánként

Most számítsuk ki az átlagos szénhidrát-, zsír-, fehérje- és rosttartalmat kategóriánként, majd jelenítsük meg ezeket kördiagramon.

In [22]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

nutritional_options = [
    ('Fehérje', 'Protein_per_serving'),
    ('Rost', 'Fiber_per_serving'),
    ('Zsír', 'Fat_g_per_serving'),
    ('Szénhidrát', 'Carbohydrate_g_per_serving')
]

nutritional_dropdown = widgets.Dropdown(
    options=[(label, col) for label, col in nutritional_options if col in df.columns],
    description='Tápanyag:',
    disabled=False,
)

def plot_nutritional_pie(selected_nutrient_col):
    clear_output(wait=True)

    if df.empty:
        print("Az adatkeret üres, nem lehet diagramot rajzolni.")
        return

    if selected_nutrient_col not in df.columns:
        print(f"Hiba: A kiválasztott oszlop ('{selected_nutrient_col}') nem található az adatkeretben.")
        return

    # Group by 'Category' and calculate the mean for the selected nutritional column
    category_nutrients = df.groupby('Category')[selected_nutrient_col].mean()

    if not category_nutrients.empty and category_nutrients.sum() > 0:
        fig, ax = plt.subplots(figsize=(10, 8))
        wedges, texts, autotexts = ax.pie(
            category_nutrients,
            labels=category_nutrients.index,
            autopct='%1.1f%%',
            startangle=90,
            wedgeprops={'edgecolor': 'black'}
        )
        # Clean up column name for title
        title_name = selected_nutrient_col.replace('_per_serving', '').replace('_g_per_serving', '').replace('_', ' ').capitalize()
        ax.set_title(f'Átlagos {title_name} eloszlás kategóriánként')
        ax.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.

        # Adjust autopct text color for better visibility
        for autotext in autotexts:
            autotext.set_color('white')

        plt.tight_layout()
        plt.show()
    else:
        print(f"Nincs elegendő adat a '{selected_nutrient_col}' kördiagram elkészítéséhez.")

# Create the interactive output widget
output = widgets.interactive_output(
    plot_nutritional_pie,
    {'selected_nutrient_col': nutritional_dropdown}
)

# Display the dropdown and the output widget
display(nutritional_dropdown, output)


Dropdown(description='Tápanyag:', options=(), value=None)

Output()